# Reproducing Moaddab & McDannald (2021) from CRCNS rrf-1

Paper: Moaddab M, McDannald MA (2021). Retrorubral field is a hub for diverse threat and
aversive outcome signals. Current Biology 31:2099-2110, doi:10.1016/j.cub.2021.02.055.

Dataset: CRCNS rrf-1, "Recordings from retrorubral field neurons in male rats undergoing fear
discrimination", doi:10.6080/K04B2ZJ5, read from the public S3 bucket crcnsarchive
(`rrf-1/live/data.zip`). CRCNS data sets are not versioned; the version is recorded as
`unversioned`. There is only one published state to pin: the bucket holds one `live` copy, with
data descriptor version 0.6 (April 23, 2021), the only version in its change history. The files
read are identified by the MD5 sums in the data set's `checksums.md5`:
`data.zip` bf5cb2951c928394084d7b67c6b4663b (the same as the S3 ETag; the notebook checks it).
Inside the zip, `data/cubeFile/cube.mat` (the authors' derived file of all units) and
`data/matFiles.tar` (one file per unit with spike and event times) are read in memory.

Claims: `claims/crcns-rrf-1/10.1016-j.cub.2021.02.055.yaml`, written from the version of record
with its supplement (Figures S1 to S6, Tables S1 to S4).

The data set also holds the paper's MATLAB code (`cubeScriptPR25.m`, `rrfAnalysis.m`). The
claims file's last check asks an author who finds the code to follow it where it settles an
open reading; the helper `rrf.py` cites the lines followed.

| Claim | Tested here |
|---|---|
| units-and-sessions | tested |
| cue-responsive-units | tested |
| behavior-discrimination | tested |
| kmeans-elbow | tested |
| kmeans-clusters | tested |
| cluster-firing-characteristics | tested |
| k1-k3-low-baseline | tested |
| table1-cue-interaction | tested |
| fig3b-excitatory-clusters | tested |
| fig3d-inhibitory-clusters | tested |
| figs4-onset-and-ramp | tested |
| fig4abc-threat-bias | tested |
| fig4def-danger-uncertainty-correlation | tested |
| fig4ghi-danger-safety-correlation | tested |
| fig4-correlations-differ | tested |
| fig5a-tuning-peaks-exaggerated | tested |
| fig5a-tuning-peaks-inhibited | tested |
| fig5a-tuning-anova | tested |
| fig5e-peak-probability-over-fear-output | tested |
| fig5-regressor-cluster | tested |
| table-s3-cluster-regression | tested |
| fig5f-fear-output-vs-peak-probability | tested |
| table1-outcome-anova | tested |
| fig6a-k1-shock to fig6h-k8-ppe-then-unsigned (7 claims) | tested |

In [ ]:
import collections
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.cluster import KMeans

import rrf

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)


def seeded(name):
    """One generator per claim section, seeded from the claim id."""
    return np.random.default_rng(sum(ord(ch) * (i + 1) for i, ch in enumerate(name)))


z = rrf.open_zip()
head = z.fp.fs.info(rrf.ZIP_URL)
print("data.zip size", head["size"], "ETag", head.get("ETag"))
assert head.get("ETag", "").strip('"') == rrf.ZIP_MD5
print([i.filename for i in z.infolist()])
cube = rrf.load_cube(z)
units = rrf.load_units(z)
print(len(units), "unit files;", len(cube["name"]), "units in cube.mat")

## Data check

What is loaded, and checks of the description against the values: counts of rats, sessions
and units; the trial table (16 trials per session, 4/2/6/4 by type); that the authors' cube
matches the spike times of the unit files; shock timing; units and ranges of the variables
used; duplicate recordings; and that spike times share the clock of the trial events.

In [ ]:
names = cube["name"]
session = np.array([n[:16] for n in names])     # rat, date, behavior program
rat = np.array([n[:4] for n in names])
print("units", len(names), "unique names", len(set(names)))
print("sessions", len(set(session)), "rats", len(set(rat)))
print(pd.DataFrame({"units": pd.Series(rat).value_counts().sort_index(),
                    "sessions": pd.Series(session).groupby(rat).nunique().sort_index()}).T)
print("program letters", collections.Counter(n[15] for n in names))
print("trial tags identical in every unit:", bool((cube["tag"][:, 0, :] == rrf.TRIAL_TYPE[:, None]).all()),
      "counts", collections.Counter(rrf.TRIAL_TYPE))

# Rebuild the cube's arrays from the unit files' spike and event times (cubeScriptPR25.m)
by_name = {u["filename"] + u["neuron"]: u for u in units}
mismatch = collections.Counter()
shock_lat, iti, odd_shock = [], [], []
for i, n in enumerate(names):
    u = by_name[n]
    on, tg = rrf.trial_onsets(u)
    if not (np.allclose(on, cube["tag"][:, 1, i]) and (tg == cube["tag"][:, 0, i]).all()):
        mismatch["trial onsets or tags"] += 1
        continue
    for key, w, nb in (("ms250", 0.25, 200), ("s1", 1.0, 50)):
        if not np.array_equal(rrf.binned_rates(u["spike"], on, w, nb), cube[f"fire.raw.{key}.interval"][:, :, i]):
            mismatch[f"{key} rates"] += 1
    if not np.array_equal(rrf.binned_rates(u["spike"], on, 2, 1, start=-2)[:, 0], cube["fire.raw.base2"][:, i]):
        mismatch["2-s baseline"] += 1
    if not np.allclose(u["wave"], cube["wave"][:, i]):
        mismatch["waveform"] += 1
    if np.any(np.diff(u["spike"]) < 0):
        mismatch["unsorted spikes"] += 1
    sh = u.get("EVT01", np.array([]))
    for t, ty in zip(on, tg):
        d = sh[(sh > t) & (sh <= t + 30)] - t
        if ty in (1, 2):
            shock_lat.extend(d)
            if len(d) != 1 or abs(d[0] - 12) > 0.01:
                odd_shock.append((n, ty, np.round(d, 3)))
    iti.extend(np.diff(np.sort(on)))
print("mismatches between cube.mat and the unit files:", dict(mismatch) or "none")
shock_lat = np.array(shock_lat)
print(f"shock onset after cue onset: {len(shock_lat)} shocks, median {np.median(shock_lat):.3f} s, "
      f"range {shock_lat.min():.3f} to {shock_lat.max():.3f} s")
print("trials whose shock is not one event at 12 s:", odd_shock)
print(f"inter-trial interval (onset to onset): mean {np.mean(iti)/60:.2f} min, "
      f"range {np.min(iti):.0f} to {np.max(iti):.0f} s")
wave = cube["wave"]
print(f"waveforms: {wave.shape[0]} samples per unit, values {wave.min():.4f} to {wave.max():.4f} (units not stated)")
r250 = cube["fire.raw.ms250.interval"]
print(f"250-ms rates: {r250.min():.0f} to {r250.max():.0f} Hz; 1-s rates {cube['fire.raw.s1.interval'].max():.0f} Hz max")
cer = cube["cer.cue"]
print(f"suppression ratio per trial: {np.nanmin(cer):.2f} to {np.nanmax(cer):.2f}, NaN {np.isnan(cer).sum()} "
      f"(the code sets NaN, no pokes in baseline or cue, to 1)")
print(f"baseline poke rate (pokes/min, 20 s before cue): {cube['poke.base'].min()} to {cube['poke.base'].max()}")

In [ ]:
# The z-scored arrays of cube.mat are the code's normalization of its raw arrays
zs1 = rrf.z_code(cube["fire.raw.s1.interval"].astype(float), cube["fire.raw.base2"].astype(float))
z250 = rrf.z_code(cube["fire.raw.ms250.interval"].astype(float), cube["fire.raw.base2"].astype(float))
print("max |z recomputed - cube z|: 1-s", float(np.nanmax(np.abs(zs1 - cube["fire.z.s1.interval"]))),
      "250-ms", float(np.nanmax(np.abs(z250 - cube["fire.z.ms250.interval"]))))

# Suppression ratios are the same for all units of a session
sr_spread = pd.DataFrame(cer.T).groupby(session).agg(lambda x: np.nanmax(x) - np.nanmin(x)).to_numpy()
print("largest within-session difference in a trial's suppression ratio:", np.nanmax(sr_spread))

# Duplicate recordings: identical spike trains, and sessions with identical event times
spk = collections.defaultdict(list)
ev = collections.defaultdict(set)
for u in units:
    spk[hash(u["spike"].tobytes())].append(u["filename"] + u["neuron"])
    ev[hash(np.concatenate([u[k] for k in sorted(u) if k.startswith("EVT")]).tobytes())].add(u["filename"])
dup_units = [v for v in spk.values() if len(v) > 1]
dup_sessions = [sorted(v) for v in ev.values() if len(v) > 1]
print("units with identical spike trains:", dup_units or "none")
print("different session names with identical event times:", dup_sessions or "none")
same_events_sessions = len(ev)
print("distinct event sets", same_events_sessions, "for", len(set(session)), "session names")

In [ ]:
# The unit files hold spikes only from 20 s before to about 33.5 s after each cue onset: the
# recording was paused between trials (event EVT14, "Pause recording" in cubeScriptPR25.m, comes
# 31.5 s after each cue onset). The descriptor does not say so.
rel_lo, rel_hi = [], []
for u in units:
    on = np.sort(rrf.trial_onsets(u)[0])
    k = np.clip(np.searchsorted(on, u["spike"] + 20.01) - 1, 0, len(on) - 1)
    rel = u["spike"] - on[k]
    rel_lo.append(rel.min()); rel_hi.append(rel.max())
print(f"spike times relative to the trial's cue onset: from {min(rel_lo):.2f} to {max(rel_hi):.2f} s")
ev14 = np.concatenate([np.sort(u["EVT14"]) - np.sort(rrf.trial_onsets(u)[0]) for u in units])
print(f"EVT14 after cue onset: {ev14.min():.3f} to {ev14.max():.3f} s")

# Clock check: spikes against the trial table. For each unit, danger against safety spike count
# (t test over trials) in the 12 s from cue onset; in the 12 s from 20 s before cue onset (no cue);
# and from cue onset with the trial labels moved one trial later in time (each trial takes the
# type of the trial before it).
def frac_discriminating(start, shift_labels=False):
    hits = []
    for i, n in enumerate(names):
        u = by_name[n]
        on = cube["tag"][:, 1, i]
        lab = rrf.TRIAL_TYPE.copy()
        if shift_labels:
            order = np.argsort(on)
            lab[order[1:]] = lab[order[:-1]]
            lab[order[0]] = 0
        t0 = on + start
        cnt = np.diff(np.searchsorted(u["spike"], np.stack([t0, t0 + 12]).T, side="right"), axis=1)[:, 0]
        p = stats.ttest_ind(cnt[lab == 1], cnt[lab == 4]).pvalue
        hits.append(np.nan_to_num(p, nan=1.0) < 0.05)
    return np.mean(hits)

for label, start, sh in (("cue onset to +12 s, true labels", 0.0, False),
                         ("-20 to -8 s, true labels", -20.0, False),
                         ("cue onset to +12 s, labels shifted one trial", 0.0, True)):
    print(f"{label}: {100 * frac_discriminating(start, sh):.1f}% of 743 units differ between danger "
          "and safety (p < 0.05)")
pk = cube["poke.s1.interval"].astype(float)
first = ~pd.Series(session).duplicated().to_numpy()      # one unit per session
pk_cue, pk_pre = pk[:, 20:30, first].mean(1), pk[:, 0:20, first].mean(1)
print("nose pokes per min, mean over 228 sessions: danger cue", round(float(pk_cue[rrf.DANGER].mean()), 1),
      "safety cue", round(float(pk_cue[rrf.SAFETY].mean()), 1), "20 s before cue", round(float(pk_pre.mean()), 1))

## units-and-sessions

> The paper recorded 743 single units from 11 rats over 228 fear discrimination sessions.

In [ ]:
n_units, n_sessions, n_rats = len(names), len(set(session)), len(set(rat))
print(f"units {n_units} (expected 743 +/- 74), sessions {n_sessions} (228 +/- 23), rats {n_rats} (11 exactly)")
print("units per session: median", int(np.median(pd.Series(session).value_counts())),
      "range", pd.Series(session).value_counts().min(), "to", pd.Series(session).value_counts().max())

**Result: reproduced.** The data set holds 743 units from 228 sessions of 11 rats, the three
counts of the paper. No unit or session is deposited twice (data check).

## cue-responsive-units

> 423 units, from 190 sessions, were cue responsive, with no fewer than 11 per rat.

The screen of the criterion (STAR Methods, Identifying cue-responsive neurons), run as the
code runs it (rrfAnalysis.m 5-60): the rates are the cube's floored integer rates, and a test
that returns NaN (no variance) counts as not significant. A variant uses unfloored rates
from the spike times.

In [ ]:
def cue_screen(base10, onset, last5):
    sig = []
    for rows in (rrf.DANGER, rrf.UNC, rrf.SAFETY):
        for x in (onset, last5):
            p = stats.ttest_rel(base10[rows], x[rows], axis=0).pvalue
            sig.append(np.nan_to_num(p, nan=1.0) < 0.05)
    return np.array(sig).T

sig = cue_screen(cube["fire.raw.base10"].astype(float), cube["fire.raw.select.cueOnset"].astype(float),
                 cube["fire.raw.select.cueLastFive"].astype(float))
resp = sig.any(1)
per_rat = pd.Series(rat[resp]).value_counts().sort_index()
paper_s1 = {"rb01": 35, "rb03": 53, "rb04": 24, "rb06": 59, "rb07": 29, "rb09": 33, "rb10": 18,
            "rb11": 11, "rb12": 47, "rb13": 69, "rb14": 45}
print(f"cue-responsive units {resp.sum()} (expected 423 +/- 42); sessions {len(set(session[resp]))} "
      f"(190 +/- 19); fewest per rat {per_rat.min()} (11 +/- 1.1)")
print(pd.DataFrame({"here": per_rat, "Figure S1": pd.Series(paper_s1)}).T)
print("units significant in each of the six tests (D onset, D last 5 s, U onset, U last 5 s, S onset, S last 5 s):",
      sig[:, [0, 1, 2, 3, 4, 5]].sum(0))

# Variant: unfloored rates from the spike times
raw_rates = np.zeros((3, 16, len(names)))
for i, n in enumerate(names):
    u, on = by_name[n], cube["tag"][:, 1, i]
    raw_rates[0, :, i] = rrf.binned_rates(u["spike"], on, 10, 1, start=-10, floor=False)[:, 0]
    raw_rates[1, :, i] = rrf.binned_rates(u["spike"], on, 1, 1, start=0, floor=False)[:, 0]
    raw_rates[2, :, i] = rrf.binned_rates(u["spike"], on, 5, 1, start=5, floor=False)[:, 0]
resp_unfloored = cue_screen(*raw_rates).any(1)
print("variant, unfloored rates:", resp_unfloored.sum(), "units;", (resp_unfloored & resp).sum(), "in common")
R = np.flatnonzero(resp)          # indices of the 423 units used below

**Result: reproduced.** The paper's screen finds 423 cue-responsive units from 190 sessions, and
the per-rat counts equal those of Figure S1 (fewest 11, rb11).

## behavior-discrimination

> In the sessions that yielded cue-responsive units, rats fully discriminated the cues by
> suppression ratio (danger > uncertainty > safety), with a main effect of cue and 95% bootstrap
> CIs of every differential suppression ratio excluding zero.

Suppression ratio per trial is the cube's `cer.cue` (20-s baseline, 10-s cue; NaN set to 1 as
in cubeScriptPR25.m). Each session with a cue-responsive unit counts once.

In [ ]:
g = seeded("behavior-discrimination")
first_of_session = ~pd.Series(session[R]).duplicated().to_numpy()
S_idx = R[first_of_session]
sr = cube["cer.cue"][:, S_idx]
ses_df = pd.DataFrame({"rat": rat[S_idx], "D": sr[rrf.DANGER].mean(0), "U": sr[rrf.UNC].mean(0),
                       "S": sr[rrf.SAFETY].mean(0)})
print("sessions", len(ses_df))

def behaviour(per_rat_df, label):
    y = per_rat_df[["D", "U", "S"]].to_numpy()
    a = rrf.mixed_anova(y)["w1"]
    print(f"{label}: cue effect F({a[1]}, {a[2]}) = {a[0]:.2f}, p = {a[3]:.3g}, partial eta2 = {a[4]:.2f}")
    print("  means D, U, S:", np.round(y.mean(0), 3))
    res = {}
    for nm, d, expv, tol in (("D-U", y[:, 0] - y[:, 1], 0.42, 0.14), ("U-S", y[:, 1] - y[:, 2], 0.40, 0.34),
                             ("D-S", y[:, 0] - y[:, 2], 0.82, 0.12)):
        ci = rrf.boot_ci(d, g)
        res[nm] = ci
        print(f"  {nm}: mean {ci[0]:.3f} CI [{ci[1]:.3f}, {ci[2]:.3f}]; expected {expv} +/- {tol}: "
              f"{'within' if abs(ci[0] - expv) <= tol else 'outside'}")
    return a, res

per_rat_mean = ses_df.groupby("rat")[["D", "U", "S"]].mean()
print(per_rat_mean.round(3).T)
beh_main = behaviour(per_rat_mean, "per rat, mean over sessions (main)")
trials = []
for i in S_idx:
    trials.append(pd.DataFrame({"rat": rat[i], "type": rrf.TRIAL_TYPE, "sr": cube["cer.cue"][:, i]}))
trials = pd.concat(trials)
trials["cue"] = trials["type"].map({1: "D", 2: "U", 3: "U", 4: "S"})
pooled = trials.groupby(["rat", "cue"])["sr"].mean().unstack()
beh_var = behaviour(pooled, "variant, per rat over pooled trials")

**Result: reproduced.** Over the 190 sessions and 11 rats, suppression is ordered danger >
uncertainty > safety, the cue effect is significant, all three bootstrap CIs exclude zero, and
the three mean differences fall within their tolerances under both readings (numbers above).

## Normalized firing

The code's normalization (trial z-scores over all trials and bins of a unit; see the helper) is
what `cube.mat` holds and what the code analyses. The Methods describe z-scoring the trial-type
means; the claims file names two readings of that, z-a (four trial types) and z-b (three).
Both are computed here as variants where they apply. The code's version is the main reading
(the claims file asks an author who finds the code to follow it), so every outcome that uses
normalized firing, or the clusters built from it, rests on the assumption that the code, not
the Methods' wording, is what the paper did.

In [ ]:
m250 = rrf.type_means(cube["fire.z.ms250.interval"][:, :, R])        # code reading
m1 = rrf.type_means(cube["fire.z.s1.interval"][:, :, R])
raw250 = cube["fire.raw.ms250.interval"][:, :, R].astype(float)
m250_za = rrf.z_methods(raw250, n_types=4)
m250_zb = rrf.z_methods(raw250, n_types=3)
NORMS = {"code": m250, "z-a": m250_za, "z-b": m250_zb}
print({k: (round(float(np.nanmin(v["D"])), 2), round(float(np.nanmax(v["D"])), 2)) for k, v in NORMS.items()},
      "range of danger normalized firing per reading")
print("units with a NaN z-score:", {k: int(np.isnan(v["D"]).any(0).sum()) for k, v in NORMS.items()})

## kmeans-elbow

> In k-means clustering of the 423 cue-responsive units, the mean squared Euclidean distance to
> the centroid falls as the cluster number rises from 1 to 15, and the decrease diminishes after
> eight clusters.

The matrix is the code's: 1-s normalized firing, cue bins 0-10 s for danger, uncertainty and
safety, and six 1-s bins from 13 s after cue onset for the four outcome types (rrfAnalysis.m,
"K means clustering"; the code starts the outcome epochs at 13 s, 0.5 s after shock offset).
The code runs `rng(1); kmeans(aa, k)` once for each k. MATLAB's k-means with its default
generator is reproduced exactly here (`rrf.matlab_kmeans`; the next section shows it gives the
paper's eight cluster sizes), so the main reading is the code's own run. The measure is the
mean over units of the squared distance to the centroid (Methods); the code's measure, the
mean over clusters of the summed squared distances (`mean(sumd)`, `sum` at k = 2, which is
what the code plots for Figure 1E), is shown too. Variants: the claims file's readings of the
outcome start (e-a: 12 s, or 12.5 s from the 250-ms z-scores) and of "10 times" (e-b: best of
10 replicates, or the mean of 10 runs, with scikit-learn's k-means++), and the Methods'
normalizations z-a and z-b.

In [ ]:
def kmeans_matrix(start, mm1=None, mm250=None):
    """start: outcome epoch start in s after cue onset (13 code, 12 or 12.5 claims e-a)."""
    to1 = lambda a, b0, n: a[b0:b0 + 4 * n].reshape(n, 4, -1).mean(1)
    if mm1 is not None and start in (12, 13):
        b = slice(20 + start, 26 + start)                  # 1-s bin j covers (j-20, j-19] s
        parts = [mm1[t][20:30] for t in ("D", "U", "S")] + [mm1[t][b] for t in ("D", "US", "UO", "S")]
    else:                                                  # from 250-ms bins
        b0 = int(round((start + 20) * 4))
        parts = [to1(mm250[t], 80, 10) for t in ("D", "U", "S")] + [to1(mm250[t], b0, 6) for t in ("D", "US", "UO", "S")]
    return np.concatenate(parts, axis=0).T

def elbow(X, mode, seed=0):
    rows = []
    for k in range(1, 16):
        if mode == "code":
            lab, sumd = rrf.matlab_kmeans(X, k, seed=1)
            sumds = [sumd]
        else:
            fits = [KMeans(k, n_init=10, random_state=seed).fit(X)] if mode == "best of 10" else \
                   [KMeans(k, n_init=1, random_state=seed + r).fit(X) for r in range(10)]
            sumds = [np.bincount(f.labels_, weights=((X - f.cluster_centers_[f.labels_]) ** 2).sum(1), minlength=k)
                     for f in fits]
        msd = np.mean([sd.sum() / len(X) for sd in sumds])
        code_measure = np.mean([sd.sum() if k == 2 else sd.mean() for sd in sumds])
        rows.append((k, msd, code_measure))
    return pd.DataFrame(rows, columns=["k", "mean sq distance", "code mean(sumd)"]).set_index("k")

def elbow_verdict(curve):
    d = -np.diff(curve.to_numpy())           # drop from k to k+1, k = 1..14
    return bool((d > 0).all()), bool((d[7:] < d[6]).all()), d

X_main = kmeans_matrix(13, mm1=m1)
e_main = elbow(X_main, "code")
print(e_main.round(3).T)
dec, after, d = elbow_verdict(e_main["mean sq distance"])
print(f"main (code's runs, Methods measure): decreasing {dec}; every drop after 8 clusters below the 7->8 drop {after}")
print("  drops k -> k+1:", np.round(d, 3))
dec_c, after_c, d_c = elbow_verdict(e_main["code mean(sumd)"])
print(f"code's measure: decreasing {dec_c}; criterion {after_c}; drops", np.round(d_c, 1))

elbow_rows = []
for nm in ("code", "z-a", "z-b"):
    for start in (13, 12, 12.5):
        X = kmeans_matrix(start, mm1=m1 if nm == "code" else None, mm250=NORMS[nm])
        for mode in ("code", "best of 10", "mean of 10 runs"):
            dec_v, after_v, d_v = elbow_verdict(elbow(X, mode)["mean sq distance"])
            elbow_rows.append((nm, start, mode, dec_v, after_v, round(d_v[6], 3), round(d_v[7:].max(), 3)))
elbow_tab = pd.DataFrame(elbow_rows, columns=["normalization", "outcome start (s)", "runs", "decreasing",
                                              "criterion met", "drop 7->8", "largest later drop"])
print(elbow_tab.to_string(index=False))
print("readings meeting the criterion:", int(elbow_tab["criterion met"].sum()), "of", len(elbow_tab))
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(e_main.index, e_main["mean sq distance"], "o-")
ax.set_xlabel("clusters"); ax.set_ylabel("mean squared distance to centroid")
plt.show()
print("plotted range:", round(float(e_main["mean sq distance"].min()), 3), "to",
      round(float(e_main["mean sq distance"].max()), 3))

**Result: reproduced, resting on an assumption.** Under the code's own runs (`rng(1)` before
each k) and the measure the code plots for Figure 1E (`mean(sumd)`), the curve falls at every
step from 1 to 15 clusters and every drop after eight clusters is smaller than the drop from 7 to
8 (numbers above). With the Methods' wording of the measure (mean squared distance per unit),
the same runs rise once, from 12 to 13 clusters, while the drops after eight clusters are still
all smaller than the 7 to 8 drop; 21 of the 27 readings in the table meet the whole criterion.
The outcome takes the code's measure, the one behind the paper's figure.

## kmeans-clusters

> Eight k-means clusters were kept, of sizes k1 33, k2 59, k3 18, k4 63, k5 58, k6 61, k7 64 and
> k8 67.

Main reading: the code's run, `rng(1); kmeans(aa, 8)`, reproduced with `rrf.matlab_kmeans`, and
the code's own map from cluster numbers to k1..k8 (`rrf.CODE_IDX`). This recovers the paper's
labels (reading c-a of the claims file: the data set holds no label file, but its code fixes
them). The claims file's matching by firing pattern (c-b) is run on these clusters as a check,
and on scikit-learn k-means solutions as a variant.

In [ ]:
PAPER_SIZES = {"k1": 33, "k2": 59, "k3": 18, "k4": 63, "k5": 58, "k6": 61, "k7": 64, "k8": 67}
idx8, _ = rrf.matlab_kmeans(X_main, 8, seed=1)
inv = {v: k for k, v in rrf.CODE_IDX.items()}
cl = np.array([inv[i] for i in idx8])                       # paper cluster of each of the 423 units
sizes = pd.Series(cl).value_counts().reindex(rrf.CLUSTERS)
print(pd.DataFrame({"here": sizes, "paper": pd.Series(PAPER_SIZES)}).T)
within = {k: abs(sizes[k] - PAPER_SIZES[k]) <= 0.1 * PAPER_SIZES[k] for k in rrf.CLUSTERS}
print("sizes within 10%:", sum(within.values()), "of 8")

pattern_rows = []
for k in rrf.CLUSTERS:
    pr = rrf.cluster_profile(m250, cl == k)
    chk = rrf.pattern_checks(pr)[k]
    pattern_rows.append((k, int(sizes[k]), all(v for _, v in chk), "; ".join(f"{n}: {bool(v)}" for n, v in chk),
                         np.round(pr["onset"], 2), np.round(pr["whole"], 2), np.round(pr["delay"], 2)))
pat = pd.DataFrame(pattern_rows, columns=["cluster", "n", "pattern found", "tests", "onset D,U,S",
                                          "cue+delay D,U,S", "delay D,U,S"])
print(pat.drop(columns="tests").to_string(index=False))
for r in pattern_rows:
    print(r[0], r[3])
match, score = rrf.match_clusters(m250, idx8)
print("pattern matching (c-b) of the code's clusters:", {k: inv[v] for k, v in match.items()},
      "agrees with the code's map for", sum(inv[v] == k for k, v in match.items()), "of 8")

# Variant: the code's k-means run on the Methods' normalizations (z-a, z-b), matched by pattern
for nm in ("z-a", "z-b"):
    lab_v, _ = rrf.matlab_kmeans(kmeans_matrix(13, mm250=NORMS[nm]), 8, seed=1)
    mp, sc = rrf.match_clusters(NORMS[nm], lab_v)
    found = list(np.unique(lab_v))
    allpat = all(sc[found.index(mp[k]), rrf.CLUSTERS.index(k)] == 1 for k in rrf.CLUSTERS)
    sz = {k: int((lab_v == mp[k]).sum()) for k in rrf.CLUSTERS}
    print(f"{nm}: sizes {sz}; all patterns {allpat}; sizes within 10% "
          f"{sum(abs(sz[k] - PAPER_SIZES[k]) <= 0.1 * PAPER_SIZES[k] for k in rrf.CLUSTERS)} of 8")

# Variant: scikit-learn k-means (k-means++, other generator), matched by pattern
var_rows = []
for mode, n_init in (("single run", 1), ("best of 10", 10)):
    for seed in range(50):
        lab = KMeans(8, n_init=n_init, random_state=seed).fit(X_main).labels_
        mp, sc = rrf.match_clusters(m250, lab)
        found = list(np.unique(lab))
        allpat = all(sc[found.index(mp[k]), rrf.CLUSTERS.index(k)] == 1 for k in rrf.CLUSTERS)
        nwithin = sum(abs((lab == mp[k]).sum() - PAPER_SIZES[k]) <= 0.1 * PAPER_SIZES[k] for k in rrf.CLUSTERS)
        var_rows.append((mode, seed, allpat, nwithin))
var = pd.DataFrame(var_rows, columns=["runs", "seed", "all patterns", "sizes within 10%"])
print(var.groupby("runs").agg(all_patterns=("all patterns", "mean"),
                              median_sizes_within=("sizes within 10%", "median"),
                              reproduced=("sizes within 10%", lambda s: np.mean((s == 8) & var.loc[s.index, "all patterns"]))))

**Result: reproduced.** The code's k-means run, repeated outside MATLAB, gives clusters of
exactly the paper's sizes (33, 59, 18, 63, 58, 61, 64, 67), and with the code's map each
cluster shows its pattern by the tests printed above. Other k-means runs (scikit-learn, other
seeds) give different partitions, rarely of these sizes: the eight clusters depend on the run.
All later claims use the code's clusters.

## cluster-firing-characteristics

> Clusters differed in baseline firing rate, coefficient of variance and coefficient of
> skewness, but not in anterior-posterior location, waveform half-duration or waveform
> amplitude ratio.

Measures as the code computes them: baseline rate the mean over trials of the floored 10-s
pre-cue rate; CV and skewness from the inter-spike intervals of the 20 s before each cue, pooled
over the 16 trials (cube.mat `fire.CV`, `fire.CS`, cubeScriptPR25.m; the claims file's main
reading); half duration and amplitude ratio from the mean waveform (rrfAnalysis.m); bregma level
per rat (rrfAnalysis.m). The variant averages CV and skewness over trials.

In [ ]:
from statsmodels.multivariate.manova import MANOVA

wf = np.array([rrf.waveform_measures(cube["wave"][:, i]) for i in R])
char = pd.DataFrame({
    "cluster": cl,
    "baseline": cube["fire.raw.base10"][:, R].astype(float).mean(0),
    "CV": cube["fire.CV"][R], "CS": cube["fire.CS"][R],
    "bregma": [rrf.BREGMA[int(rat[i][2:])] for i in R],
    "half_duration": wf[:, 0], "amplitude_ratio": wf[:, 1]})
print(char.describe().round(3).T)
print(char.groupby("cluster").mean().round(3))

def char_tests(df, label):
    print(label)
    mv = MANOVA.from_formula("baseline + CV + CS + bregma + half_duration + amplitude_ratio ~ C(cluster)", data=df)
    wilks = mv.mv_test().results["C(cluster)"]["stat"].loc["Wilks' lambda"]
    print(f"  MANOVA Wilks' lambda {wilks['Value']:.3f}, F({wilks['Num DF']:.0f}, {wilks['Den DF']:.0f}) = "
          f"{wilks['F Value']:.2f}, p = {wilks['Pr > F']:.3g}")
    out = {}
    for m in ("baseline", "CV", "CS", "bregma", "half_duration", "amplitude_ratio"):
        groups = [df.loc[df.cluster == k, m].dropna() for k in rrf.CLUSTERS]
        F, p = stats.f_oneway(*groups)
        ss_b = sum(len(g_) * (g_.mean() - df[m].mean()) ** 2 for g_ in groups)
        ss_t = ((df[m] - df[m].mean()) ** 2).sum()
        out[m] = (F, p)
        print(f"  {m}: F(7, {len(df) - 8}) = {F:.2f}, p = {p:.3g}, eta2 = {ss_b / ss_t:.3f}")
    return out

ch_main = char_tests(char, "main: ISIs pooled over the 16 trials")
# Variant: CV and skewness per trial, then averaged
cv_t, cs_t = [], []
for i in R:
    u, on = by_name[names[i]], cube["tag"][:, 1, i]
    cvs, css = [], []
    for t in on:
        isi = np.diff(u["spike"][(u["spike"] > t - 20) & (u["spike"] <= t)])
        if len(isi) > 2:
            cvs.append(isi.std(ddof=1) / isi.mean())
            css.append(3 * (isi.mean() - np.median(isi)) / isi.std(ddof=1))
    cv_t.append(np.mean(cvs) if cvs else np.nan); cs_t.append(np.mean(css) if css else np.nan)
char_v = char.assign(CV=cv_t, CS=cs_t).dropna()
print("variant drops", len(char) - len(char_v), "units with too few spikes")
ch_var = char_tests(char_v, "variant: CV and skewness averaged over trials")
def pooled_isi(i):
    sp, on = by_name[names[i]]["spike"], cube["tag"][:, 1, i]
    return np.concatenate([np.diff(sp[(sp > t - 20) & (sp <= t)]) for t in on])

cv_check = np.array([pooled_isi(i).std(ddof=1) / pooled_isi(i).mean() for i in R])
print("cube CV against CV recomputed from spike times, max |difference| over 423 units:",
      float(np.nanmax(np.abs(cv_check - cube["fire.CV"][R]))))

**Result: reproduced.** Baseline rate, CV and skewness differ between clusters and location,
half duration and amplitude ratio do not (p values above), under both readings of the
inter-spike intervals.

## k1-k3-low-baseline

> Units of clusters k1 and k3 had lower baseline firing rates than all other clusters.

In [ ]:
low = char.cluster.isin(["k1", "k3"])
t_res = stats.ttest_ind(char.baseline[low], char.baseline[~low])
print(f"k1 + k3: n = {low.sum()}, mean {char.baseline[low].mean():.2f} Hz; others n = {(~low).sum()}, "
      f"mean {char.baseline[~low].mean():.2f} Hz; t({len(char) - 2}) = {t_res.statistic:.2f}, p = {t_res.pvalue:.3g}")
print("Welch variant p =", stats.ttest_ind(char.baseline[low], char.baseline[~low], equal_var=False).pvalue)
print(char.groupby("cluster").baseline.agg(["mean", "median", "count"]).round(2).T)

**Result: reproduced.** k1 and k3 units have the lower baseline rate, and the t test is
significant (numbers above).

## table1-cue-interaction

> Every cluster showed a significant trial type x interval interaction for cue firing.

Per cluster, the 56 250-ms bins from 2 s before cue onset to the end of the delay (bins 73-128,
as the code's ANOVA data), danger, uncertainty and safety; repeated measures with trial type and
interval within units, uncorrected (Greenhouse-Geisser as a variant).

In [ ]:
def cue_anova_table(mm, label):
    rows = []
    for k in rrf.CLUSTERS:
        sel = cl == k
        y = np.stack([mm[t][rrf.CUE_ANOVA][:, sel] for t in "DUS"], axis=0).transpose(2, 0, 1)
        a = rrf.mixed_anova(y)["w1 x w2"]
        rows.append((k, sel.sum(), f"F({a[1]}, {a[2]}) = {a[0]:.2f}", a[3], round(a[4], 3), round(a[5], 3), a[6]))
    tab = pd.DataFrame(rows, columns=["cluster", "n", "interaction", "p", "partial eta2", "GG eps", "GG p"])
    print(label); print(tab.to_string(index=False))
    return tab

t1cue = cue_anova_table(m250, "code normalization")
print("clusters with p < 0.05:", int((t1cue.p < 0.05).sum()), "of 8")
for nm in ("z-a", "z-b"):
    tv = cue_anova_table(NORMS[nm], nm)
    print(nm, "clusters with p < 0.05:", int((tv.p < 0.05).sum()))

**Result: reproduced.** The trial type x interval interaction is significant in all eight
clusters, with or without the Greenhouse-Geisser correction and under each normalization; for
k1 the degrees of freedom are those of Table 1, F(110, 3520).

## fig3b-excitatory-clusters and fig3d-inhibitory-clusters

> Clusters k1, k3, k5 and k7 show excitatory firing to danger and uncertainty; k1 fires most to
> danger at cue onset with lower, similar firing to uncertainty and safety; k3 (whole cue and
> delay), k5 (late cue and delay) and k7 (delay) fully differentiate danger > uncertainty > safety.

> Clusters k2, k4, k6 and k8 show sustained inhibition through the cue and delay; k2 is inhibited
> to uncertainty only, k4 equally to danger and uncertainty, k6 most to danger, less to
> uncertainty, with excitation to safety, and k8 to danger only.

Per unit, mean normalized firing to each cue in the cluster's Figure 3 interval (the code's
bins: k1 81-84, k5 101-128, k7 121-128, others 81-128); per-unit differences danger -
uncertainty and uncertainty - safety with studentized bootstrap CIs (Table S1).

In [ ]:
TABLE_S1 = {  # mean, CI low, CI high, tolerance (claims file)
    "k1": [(0.51, 0.30, 0.74, 0.23), (0.25, -0.11, 0.59, 0.36)],
    "k3": [(0.73, 0.37, 0.95, 0.36), (0.62, 0.38, 0.80, 0.24)],
    "k5": [(0.33, 0.20, 0.45, 0.13), (0.35, 0.24, 0.44, 0.11)],
    "k7": [(0.21, 0.08, 0.33, 0.13), (0.27, 0.17, 0.37, 0.10)],
    "k2": [(0.27, 0.20, 0.33, 0.07), (-0.30, -0.37, -0.23, 0.07)],
    "k4": [(0.01, -0.05, 0.07, 0.06), (-0.50, -0.56, -0.44, 0.06)],
    "k6": [(-0.23, -0.30, -0.15, 0.08), (-0.37, -0.44, -0.29, 0.08)],
    "k8": [(-0.45, -0.51, -0.38, 0.07), (0.06, -0.01, 0.13, 0.07)],
}

def table_s1(clusters, mm, gen, label):
    rows = []
    for k in clusters:
        sel = cl == k
        per = {t: np.nanmean(mm[t][rrf.FIG3_INTERVAL[k]][:, sel], axis=0) for t in "DUS"}
        for nm, d, ref in (("D-U", per["D"] - per["U"], TABLE_S1[k][0]), ("U-S", per["U"] - per["S"], TABLE_S1[k][1])):
            ci = rrf.boot_ci(d, gen)
            paper_sign = rrf.ci_sign((ref[0], ref[1], ref[2]))
            rows.append((k, nm, round(ci[0], 3), round(ci[1], 3), round(ci[2], 3), ref[0], f"[{ref[1]}, {ref[2]}]",
                         rrf.ci_sign(ci) == paper_sign, abs(ci[0] - ref[0]) <= ref[3]))
    tab = pd.DataFrame(rows, columns=["cluster", "diff", "mean", "low", "high", "paper mean", "paper CI",
                                      "CI agrees", "mean within tol"])
    print(label); print(tab.to_string(index=False))
    return tab

In [ ]:
g = seeded("fig3b-excitatory-clusters")
s1_exc = table_s1(("k1", "k3", "k5", "k7"), m250, g, "code normalization")
print("CIs agreeing:", int(s1_exc["CI agrees"].sum()), "of 8; means within tolerance:", int(s1_exc["mean within tol"].sum()), "of 8")
print("clusters with both CIs agreeing:", int(s1_exc.groupby("cluster")["CI agrees"].all().sum()), "of 4")
for nm in ("z-a", "z-b"):
    tv = table_s1(("k1", "k3", "k5", "k7"), NORMS[nm], seeded("fig3b-excitatory-clusters"), nm)
    print(nm, "CIs agreeing", int(tv["CI agrees"].sum()), "means within", int(tv["mean within tol"].sum()))
for k in ("k1", "k3", "k5", "k7"):
    pr = rrf.cluster_profile(m250, cl == k)
    print(k, "mean D, U, S in its interval:", np.round([np.nanmean(m250[t][rrf.FIG3_INTERVAL[k]][:, cl == k]) for t in "DUS"], 3))

**Result: reproduced.** With the code's normalization all eight CIs agree with Table S1 on
excluding zero and on sign (k1's uncertainty - safety includes zero, as in the paper; the
other seven exclude zero, positive), and all eight means are within tolerance.

In [ ]:
g = seeded("fig3d-inhibitory-clusters")
s1_inh = table_s1(("k2", "k4", "k6", "k8"), m250, g, "code normalization")
print("CIs agreeing:", int(s1_inh["CI agrees"].sum()), "of 8; means within tolerance:", int(s1_inh["mean within tol"].sum()), "of 8")
print("clusters with both CIs agreeing:", int(s1_inh.groupby("cluster")["CI agrees"].all().sum()), "of 4")
for nm in ("z-a", "z-b"):
    tv = table_s1(("k2", "k4", "k6", "k8"), NORMS[nm], seeded("fig3d-inhibitory-clusters"), nm)
    print(nm, "CIs agreeing", int(tv["CI agrees"].sum()), "means within", int(tv["mean within tol"].sum()))
for k in ("k2", "k4", "k6", "k8"):
    print(k, "mean D, U, S in 0-12 s:", np.round([np.nanmean(m250[t][rrf.FIG3_INTERVAL[k]][:, cl == k]) for t in "DUS"], 3))

**Result: reproduced.** All eight CIs agree with Table S1 and all eight means are within
tolerance (numbers above).

## figs4-onset-and-ramp

> From cue onset to delay, k1 firing decreases for all three cues (phasic onset firing), k5
> firing increases for danger and uncertainty (ramping), and k2 firing increases for danger.

Per unit and cue, delay (bins 121-128) minus onset (bins 81-84), as the code's `tempDiff`.

In [ ]:
g = seeded("figs4-onset-and-ramp")
S4 = {("k1", "D"): (-1.32, 0.35, -1), ("k1", "U"): (-0.95, 0.32, -1), ("k1", "S"): (-0.52, 0.32, -1),
      ("k5", "D"): (0.50, 0.18, 1), ("k5", "U"): (0.23, 0.12, 1), ("k2", "D"): (0.18, 0.13, 1)}
rows = []
for k in rrf.CLUSTERS:
    for t in "DUS":
        d = np.nanmean(m250[t][rrf.DELAY][:, cl == k], 0) - np.nanmean(m250[t][rrf.ONSET][:, cl == k], 0)
        ci = rrf.boot_ci(d, g)
        ref = S4.get((k, t))
        rows.append((k, t, round(ci[0], 3), round(ci[1], 3), round(ci[2], 3), rrf.ci_sign(ci),
                     ref[0] if ref else None, (rrf.ci_sign(ci) == ref[2]) if ref else None,
                     (abs(ci[0] - ref[0]) <= ref[1]) if ref else None))
s4 = pd.DataFrame(rows, columns=["cluster", "cue", "mean", "low", "high", "CI sign", "paper mean", "CI agrees", "within tol"])
print(s4.to_string(index=False))
claimed = s4.dropna(subset=["paper mean"])
print("claimed CIs with the paper's sign:", int(claimed["CI agrees"].astype(bool).sum()), "of 6; means within tolerance:",
      int(claimed["within tol"].astype(bool).sum()), "of 6")
for nm in ("z-a", "z-b"):
    gv = seeded("figs4-onset-and-ramp")
    agree = within_ = 0
    for (k, t), ref in S4.items():
        mm = NORMS[nm]
        d = np.nanmean(mm[t][rrf.DELAY][:, cl == k], 0) - np.nanmean(mm[t][rrf.ONSET][:, cl == k], 0)
        ci = rrf.boot_ci(d, gv)
        agree += rrf.ci_sign(ci) == ref[2]; within_ += abs(ci[0] - ref[0]) <= ref[1]
    print(f"{nm}: claimed CIs with the paper's sign {agree} of 6; means within tolerance {within_} of 6")

**Result: reproduced.** The six claimed changes have CIs that exclude zero with the paper's sign
and means within tolerance (table above); the other cues and clusters are listed for comparison
with Figure S4.

## fig4abc-threat-bias

> Except for k2, each cluster's uncertainty firing has the direction and time course of its
> danger firing at lower magnitude, and safety firing tends to oppose danger firing; k4 and k6 are
> inhibited to danger but excited to safety, k3 and k5 the reverse.

In [ ]:
rows = []
for k in rrf.CLUSTERS:
    dus = [np.nanmean(m250[t][rrf.FIG3_INTERVAL[k]][:, cl == k]) for t in "DUS"]
    s5 = {p: np.round([np.nanmean(m250[t][sl][:, cl == k]) for t in "DUS"], 2)
          for p, sl in (("onset", rrf.ONSET), ("late", rrf.LATE), ("delay", rrf.DELAY))}
    rows.append((k, *np.round(dus, 3), np.sign(dus[1]) == np.sign(dus[0]) and abs(dus[1]) < abs(dus[0]),
                 np.sign(dus[2]) != np.sign(dus[0]), s5["onset"], s5["late"], s5["delay"]))
bias = pd.DataFrame(rows, columns=["cluster", "D", "U", "S", "U like D, smaller", "S opposes D",
                                   "onset D,U,S", "late D,U,S", "delay D,U,S"])
print(bias.to_string(index=False))
b = bias.set_index("cluster")
part1 = bool(b.loc["k4", "D"] < 0 < b.loc["k4", "S"] and b.loc["k6", "D"] < 0 < b.loc["k6", "S"] and
             b.loc["k3", "D"] > 0 > b.loc["k3", "S"] and b.loc["k5", "D"] > 0 > b.loc["k5", "S"])
part2 = int(b.drop("k2")["U like D, smaller"].sum())
print("(1) k4, k6 danger < 0 < safety and k3, k5 danger > 0 > safety:", part1)
print("(2) clusters other than k2 with uncertainty of danger's sign and smaller:", part2, "of 7")
print("safety opposes danger in", int(b["S opposes D"].sum()), "of 8 clusters")
for nm in ("z-a", "z-b"):
    mm = NORMS[nm]
    bb = {k: [np.nanmean(mm[t][rrf.FIG3_INTERVAL[k]][:, cl == k]) for t in "DUS"] for k in rrf.CLUSTERS}
    p1 = bool(bb["k4"][0] < 0 < bb["k4"][2] and bb["k6"][0] < 0 < bb["k6"][2] and bb["k3"][0] > 0 > bb["k3"][2]
              and bb["k5"][0] > 0 > bb["k5"][2])
    p2 = sum(np.sign(bb[k][1]) == np.sign(bb[k][0]) and abs(bb[k][1]) < abs(bb[k][0]) for k in rrf.CLUSTERS if k != "k2")
    print(f"{nm}: part (1) {p1}; part (2) {p2} of 7")

**Result: partially reproduced.** Part (1) holds: k4 and k6 are inhibited to danger and excited to
safety, k3 and k5 the reverse. Part (2) holds in five of the seven clusters other than k2, one
short of the six the criterion asks: in k4 uncertainty inhibition equals danger inhibition
(-0.324 against -0.319, as the paper itself reports k4 as equal), and in k7 mean uncertainty
firing over the delay is about zero (-0.002 against danger 0.207). Safety firing has the
opposite sign to danger in five of eight clusters.

## fig4def-danger-uncertainty-correlation, fig4ghi-danger-safety-correlation, fig4-correlations-differ

> Across all cue-responsive units, danger and uncertainty firing are positively related at cue
> onset (R2 = 0.48), late cue (R2 = 0.21) and delay (R2 = 0.21).

> Danger and safety firing are positively related at cue onset (R2 = 0.14) but negatively related
> in the late cue (R2 = 0.03) and delay (R2 = 0.02).

> The danger-uncertainty and danger-safety correlations differ from one another in each period
> (Z = 7.69 onset, 8.94 late cue, 8.20 delay).

Per unit, mean normalized firing in the first 1 s of the cue, the last 5 s and the 2-s delay
(the code's bins); Pearson correlation over the 423 units (`corr` in rrfAnalysis.m). Meng's Z
uses the uncertainty-safety correlation of the same units as the correlation between the two
predictors.

In [ ]:
PERIODS = {"onset": rrf.ONSET, "late cue": rrf.LATE, "delay": rrf.DELAY}

def correlations(mm):
    rows = []
    for p, sl in PERIODS.items():
        D, U, S = (np.nanmean(mm[t][sl], 0) for t in "DUS")
        rdu, pdu = stats.pearsonr(D, U)
        rds, pds = stats.pearsonr(D, S)
        rus = stats.pearsonr(U, S)[0]
        zm, pm = rrf.meng_z(rdu, rds, rus, len(D))
        rows.append((p, rdu, rdu ** 2, pdu, rds, rds ** 2, pds, rus, zm, pm))
    return pd.DataFrame(rows, columns=["period", "r DU", "R2 DU", "p DU", "r DS", "R2 DS", "p DS", "r US", "Meng Z", "Meng p"])

corr_tabs = {nm: correlations(mm) for nm, mm in NORMS.items()}
for nm, tab in corr_tabs.items():
    print(nm); print(tab.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

In [ ]:
exp_du = {"onset": (0.48, 0.048), "late cue": (0.21, 0.021), "delay": (0.21, 0.021)}
for nm, tab in corr_tabs.items():
    ok = [(r["r DU"] > 0 and r["p DU"] < 0.05, abs(r["R2 DU"] - exp_du[r["period"]][0]) <= exp_du[r["period"]][1])
          for _, r in tab.iterrows()]
    print(f"{nm}: R2 {np.round(tab['R2 DU'].to_numpy(), 3)}; positive and significant {[a for a, _ in ok]}; "
          f"within tolerance {[b for _, b in ok]}")

**Result: reproduced.** Under the code's normalization the three correlations are positive and
significant with R2 0.479, 0.209 and 0.210, all within tolerance of 0.48, 0.21 and 0.21. The
Methods' normalizations give positive, significant correlations with R2 printed above.

In [ ]:
exp_ds = {"onset": (0.14, 0.014, 1), "late cue": (0.03, 0.005, -1), "delay": (0.02, 0.005, -1)}
for nm, tab in corr_tabs.items():
    ok = [(np.sign(r["r DS"]) == exp_ds[r["period"]][2] and r["p DS"] < 0.05,
           abs(r["R2 DS"] - exp_ds[r["period"]][0]) <= exp_ds[r["period"]][1]) for _, r in tab.iterrows()]
    print(f"{nm}: r {np.round(tab['r DS'].to_numpy(), 3)}, R2 {np.round(tab['R2 DS'].to_numpy(), 4)}, "
          f"p {np.array([f'{v:.2g}' for v in tab['p DS']])}; sign and significance {[a for a, _ in ok]}; "
          f"within tolerance {[b for _, b in ok]}")

**Result: reproduced.** Under the code's normalization danger and safety firing are positively
correlated at onset and negatively in the late cue and delay, all with p < 0.05, and the three R2
are within tolerance (numbers above).

In [ ]:
exp_z = {"onset": 7.69, "late cue": 8.94, "delay": 8.20}
for nm, tab in corr_tabs.items():
    print(f"{nm}: Z {np.round(tab['Meng Z'].to_numpy(), 2)} (expected 7.69, 8.94, 8.20 +/- 10%); p "
          f"{np.array([f'{v:.2g}' for v in tab['Meng p']])}; within tolerance "
          f"{[abs(z_ - exp_z[p]) <= 0.1 * exp_z[p] for p, z_ in zip(tab['period'], tab['Meng Z'])]}")

**Result: reproduced.** Meng's Z under the code's normalization is within 10% of the paper's
value in all three periods, with p < 0.05 (numbers above).

## Single-unit regression (Figure 5)

Per unit and 1-s bin from 2 s before cue onset to the end of the delay (14 bins, the code's
columns 19-32 of the 1-s z-scores), the 16 trials' normalized firing regressed on a constant,
the cue's threat probability (danger 1, safety 0, uncertainty set to 0, 0.125, ... 1) and the
trial's suppression ratio for the whole cue (fear output), with MATLAB `regress`'s handling of
rank deficiency. Variants: (t-b) trial firing z-scored over the unit's 16 trials in each bin
(from the differential rates); (t-a) tuning curves from the 10 cue bins in place of the
cluster periods.

In [ ]:
z1_trials = cube["fire.z.s1.interval"][:, 18:32, :][:, :, R]                 # (16, 14, 423)
d1 = cube["fire.diff.s1.interval"][:, 18:32, :][:, :, R].astype(float)
sd1 = d1.std(0, ddof=1, keepdims=True)
z1_trials_tb = np.where(sd1 > 0, (d1 - d1.mean(0, keepdims=True)) / np.where(sd1 > 0, sd1, 1), 0.0)
fear = cube["cer.cue"][:, R]

def regressions(ztr):
    tp = np.zeros((len(R), 9, 14)); fo = np.zeros((len(R), 9, 14)); deficient = np.zeros(len(R), bool)
    for u in range(len(R)):
        for a, val in enumerate(rrf.ASSIGNMENTS):
            prob = np.r_[np.ones(4), np.full(8, val), np.zeros(4)]
            X = np.column_stack([np.ones(16), prob, fear[:, u]])
            b, rank = rrf.regress_betas(ztr[:, :, u], X)
            tp[u, a], fo[u, a] = b[1], b[2]
            deficient[u] |= rank < 3
    return tp, fo, deficient

tp_beta, fo_beta, deficient = regressions(z1_trials)
tp_beta_tb, fo_beta_tb, _ = regressions(z1_trials_tb)
print("units with a rank-deficient regression (constant suppression ratio over trials):", int(deficient.sum()),
      "; clusters", collections.Counter(cl[deficient]))
print("beta ranges: threat probability", np.round([tp_beta.min(), tp_beta.max()], 2),
      "fear output", np.round([fo_beta.min(), fo_beta.max()], 2))
SEVEN = [k for k in rrf.CLUSTERS if k != "k2"]

def tuning(tp, cols_by_cluster):
    curves, per_unit = {}, {}
    for k in SEVEN:
        sel = cl == k
        v = tp[sel][:, :, cols_by_cluster[k]].mean(2)
        if k in rrf.INVERTED:
            v = -v
        per_unit[k] = v
        curves[k] = v.mean(0)
    return curves, per_unit

TEN_CUE = {k: slice(2, 12) for k in SEVEN}
tune_main = tuning(tp_beta, rrf.TUNE_COLS)
tune_variants = {"t-a cue bins": tuning(tp_beta, TEN_CUE), "t-b trial z": tuning(tp_beta_tb, rrf.TUNE_COLS),
                 "t-a and t-b": tuning(tp_beta_tb, TEN_CUE)}
peaks = pd.DataFrame({nm: {k: rrf.ASSIGNMENTS[int(np.argmax(c[k]))] for k in SEVEN}
                      for nm, (c, _) in {"code periods, code z": tune_main, **tune_variants}.items()})
peaks["paper"] = pd.Series(rrf.PAPER_PEAK)
print(peaks)
print(pd.DataFrame(tune_main[0], index=rrf.ASSIGNMENTS).round(3).T)

## fig5a-tuning-peaks-exaggerated

> Threat tuning curves of k1, k3 and k5 peak at an uncertainty assignment of 0.375 and k7 at
> 0.500, above the actual shock probability of 0.250 (exaggerated threat probability signaling).

In [ ]:
exc = ["k1", "k3", "k5", "k7"]
print(peaks.loc[exc])
for col in peaks.columns[:-1]:
    print(f"{col}: all four match {bool((peaks.loc[exc, col] == peaks.loc[exc, 'paper']).all())}; "
          f"all above 0.25 {bool((peaks.loc[exc, col] > 0.25).all())}")
fig, ax = plt.subplots(figsize=(4, 3))
for k in SEVEN:
    ax.plot(rrf.ASSIGNMENTS, tune_main[0][k], label=k)
ax.set_xlabel("uncertainty assignment"); ax.set_ylabel("threat probability beta"); ax.legend(fontsize=6)
plt.show()
print("plotted range:", round(min(v.min() for v in tune_main[0].values()), 3), "to",
      round(max(v.max() for v in tune_main[0].values()), 3))

**Result: reproduced.** With the code's periods and normalization the tuning curves peak at 0.375
for k1, k3 and k5 and at 0.500 for k7, the paper's four values. The readings the code does not
use give other peaks for k1, k5 or k7 (table above): the cue-bin reading (t-a) moves k1 to 0.750,
k5 to 0.500 and k7 to 0.625, and the trial z-score reading (t-b) moves k1 to 0.250.

## fig5a-tuning-peaks-inhibited

> Among the cue-inhibited clusters, k4 (peak 0.875) and k6 (0.625) show even greater exaggeration
> of threat probability, while k8 peaks at the actual probability, 0.250, flattening toward 0.000.

In [ ]:
inh = ["k4", "k6", "k8"]
print(peaks.loc[inh])
for col in peaks.columns[:-1]:
    print(f"{col}: all three match {bool((peaks.loc[inh, col] == peaks.loc[inh, 'paper']).all())}")
print("k8 curve:", np.round(tune_main[0]["k8"], 3))

**Result: reproduced.** k4 peaks at 0.875, k6 at 0.625 and k8 at 0.250 under every reading, and
the k8 curve falls to -0.012 at an assignment of 1.

## fig5a-tuning-anova

> The tuning curves differ between clusters (main effect of cluster and cluster x assignment
> interaction) and show a main effect of assignment.

In [ ]:
for nm, (_, per_unit) in {"code periods, code z": tune_main, **tune_variants}.items():
    y = np.concatenate([per_unit[k] for k in SEVEN])
    grp = np.concatenate([[k] * len(per_unit[k]) for k in SEVEN])
    a = rrf.mixed_anova(y, grp)
    print(nm, f"(n = {len(y)})")
    for eff, lab_ in (("group", "cluster"), ("w1", "assignment"), ("w1 x group", "cluster x assignment")):
        F, df1, df2, p, eta = a[eff][:5]
        print(f"  {lab_}: F({df1}, {df2}) = {F:.2f}, p = {p:.3g}, partial eta2 = {eta:.3f}, GG p = {a[eff][6]:.3g}")

**Result: reproduced.** Main effect of cluster F(6, 357) = 4.14, p = 0.00049; of assignment
F(8, 2856) = 24.84, p = 2.7e-37; cluster x assignment F(48, 2856) = 7.22, p = 1.2e-43, the
paper's values. All three effects are significant under every reading.

## fig5e-peak-probability-over-fear-output

> With peak threat probability and fear output as regressors, neither is signaled before the
> cue; peak threat probability is signaled from cue onset through the delay, fear output in every
> bin but the first 1 s of the cue, and peak threat probability signaling exceeds fear output
> signaling in every cue and delay bin.

Each cluster's regression at the paper's peak assignment, in the 14 bins; for k4, k6 and k8 both
betas multiplied by -1, as the code does (reading s-a, sign-inverted, settled by the code; the
betas left as computed are a variant). The ANOVA has regressor and bin within units and cluster
between.

In [ ]:
BIN_NAMES = ["pre1", "pre2"] + [f"cue{i}" for i in range(1, 11)] + ["delay1", "delay2"]

def peak_betas(tp, fo, peaks_by_cluster, invert=True):
    rows_tp, rows_fo, grp = [], [], []
    for k in SEVEN:
        a = int(round(peaks_by_cluster[k] / 0.125))
        sign = -1.0 if (invert and k in rrf.INVERTED) else 1.0
        rows_tp.append(sign * tp[cl == k, a]); rows_fo.append(sign * fo[cl == k, a]); grp += [k] * (cl == k).sum()
    return np.concatenate(rows_tp), np.concatenate(rows_fo), np.array(grp)

def table_s2(TP, FO, gen):
    rows = []
    for b_, nm in enumerate(BIN_NAMES):
        c1, c2, c3 = rrf.boot_ci(TP[:, b_], gen), rrf.boot_ci(FO[:, b_], gen), rrf.boot_ci(TP[:, b_] - FO[:, b_], gen)
        rows.append((nm, *np.round(c1, 3), *np.round(c2, 3), *np.round(c3, 3)))
    return pd.DataFrame(rows, columns=["bin", "TP", "TP lo", "TP hi", "FO", "FO lo", "FO hi", "diff", "diff lo", "diff hi"])

S2_TP = [0.02, -0.02, 0.46, 0.45, 0.43, 0.38, 0.46, 0.50, 0.56, 0.55, 0.55, 0.52, 0.44, 0.49]
S2_TP_TOL = [0.05, 0.05, 0.12, 0.1, 0.1, 0.09, 0.09, 0.11, 0.11, 0.11, 0.11, 0.1, 0.1, 0.1]
S2_FO = [-0.01, 0.01, 0.06, 0.12, 0.14, 0.18, 0.18, 0.17, 0.10, 0.13, 0.20, 0.16, 0.17, 0.14]
S2_FO_TOL = [0.04, 0.04, 0.08, 0.09, 0.08, 0.09, 0.08, 0.09, 0.09, 0.12, 0.09, 0.1, 0.09, 0.1]

def judge_s2(tab):
    tp_inc = (tab["TP lo"] <= 0) & (tab["TP hi"] >= 0)
    fo_inc = (tab["FO lo"] <= 0) & (tab["FO hi"] >= 0)
    pattern = {
        "pre bins: both include zero": bool(tp_inc[:2].all() and fo_inc[:2].all()),
        "TP excludes zero (above) in all 12 cue and delay bins": int((tab["TP lo"][2:] > 0).sum()),
        "FO above zero in cue2..delay2 (11 bins)": int((tab["FO lo"][3:] > 0).sum()),
        "FO includes zero in cue1": bool(fo_inc[2]),
        "TP - FO above zero (12 bins)": int((tab["diff lo"][2:] > 0).sum()),
    }
    means = int(sum(abs(tab["TP"][i] - S2_TP[i]) <= S2_TP_TOL[i] for i in range(14)) +
                sum(abs(tab["FO"][i] - S2_FO[i]) <= S2_FO_TOL[i] for i in range(14)))
    return pattern, means

def regressor_anova(TP, FO, grp):
    y = np.stack([TP, FO], axis=1)            # units x regressor x bin
    return rrf.mixed_anova(y, grp)

def show_anova(a, keys):
    for eff, lab_ in keys:
        F, df1, df2, p, eta = a[eff][:5]
        print(f"  {lab_}: F({df1}, {df2}) = {F:.2f}, p = {p:.3g}, partial eta2 = {eta:.3f}, GG p = {a[eff][6]:.3g}")

In [ ]:
g = seeded("fig5e-peak-probability-over-fear-output")
TP, FO, grp = peak_betas(tp_beta, fo_beta, rrf.PAPER_PEAK)
print("units", len(TP), "in", len(SEVEN), "clusters")
s2 = table_s2(TP, FO, g)
print(s2.to_string(index=False))
pat5e, means5e = judge_s2(s2)
print(pat5e, "; means within tolerance:", means5e, "of 28")
a5 = regressor_anova(TP, FO, grp)
print("ANOVA, regressor and bin within units, cluster between:")
show_anova(a5, (("w1", "regressor"), ("w1 x w2", "regressor x bin"), ("w2", "bin"), ("group", "cluster"),
                ("w1 x group", "regressor x cluster"), ("w1 x w2 x group", "regressor x bin x cluster")))
# Variants: betas not sign-inverted (s-a), and the trial z-scores of reading t-b
for label, (tp_, fo_, inv_) in {"not inverted": (tp_beta, fo_beta, False),
                                "t-b trial z": (tp_beta_tb, fo_beta_tb, True)}.items():
    TPv, FOv, grpv = peak_betas(tp_, fo_, rrf.PAPER_PEAK, invert=inv_)
    tv = table_s2(TPv, FOv, seeded("fig5e-peak-probability-over-fear-output"))
    pv, mv = judge_s2(tv)
    av = regressor_anova(TPv, FOv, grpv)
    print(label, pv, "means within", mv,
          f"regressor F = {av['w1'][0]:.2f} p = {av['w1'][3]:.3g}; regressor x bin F = {av['w1 x w2'][0]:.2f} p = {av['w1 x w2'][3]:.3g}")

**Result: reproduced.** The CI pattern of Table S2 is found in full: neither beta excludes zero
before the cue, the peak threat probability beta is above zero in all 12 cue and delay bins,
the fear output beta in all of them but the first cue bin, and the difference is above zero in
all 12; all 28 means are within tolerance. The regressor main effect (F(1, 357) = 31.40,
p = 4.2e-8) and the regressor x bin interaction (F(13, 4641) = 4.63, p = 5.6e-8) are
significant. The error df are 357, where the paper's 355 imply two units fewer (see
table-s3-cluster-regression). With the betas left as computed (not sign-inverted) the pattern
is not found, which confirms that the paper's Table S2 uses the code's inversion.

## fig5-regressor-cluster

> Signaling varied somewhat between clusters (regressor x cluster interaction) with no regressor
> x cluster x bin interaction.

In [ ]:
show_anova(a5, (("w1 x group", "regressor x cluster"), ("w1 x w2 x group", "regressor x bin x cluster")))
for label, (tp_, fo_, inv_) in {"not inverted": (tp_beta, fo_beta, False),
                                "t-b trial z": (tp_beta_tb, fo_beta_tb, True)}.items():
    av = regressor_anova(*peak_betas(tp_, fo_, rrf.PAPER_PEAK, invert=inv_))
    print(label); show_anova(av, (("w1 x group", "regressor x cluster"), ("w1 x w2 x group", "regressor x bin x cluster")))

**Result: reproduced.** Regressor x cluster F(6, 357) = 2.40, p = 0.027 (paper 2.15, p = 0.048),
and no regressor x cluster x bin interaction, F(78, 4641) = 1.10, p = 0.25 (paper 1.06, p = 0.33).

## table-s3-cluster-regression

> Within clusters, peak threat probability exceeds fear output (main effect of regressor) in k3,
> k4, k6, k7 and k8 but not in k1 or k5, and only k7 shows a regressor x bin interaction.

In [ ]:
S3_MAIN = {"k1": False, "k3": True, "k4": True, "k5": False, "k6": True, "k7": True, "k8": True}
S3_INT = {k: k == "k7" for k in SEVEN}
rows = []
for k in SEVEN:
    sel = grp == k
    a = rrf.mixed_anova(np.stack([TP[sel], FO[sel]], axis=1))
    larger = TP[sel].mean() > FO[sel].mean()
    rows.append((k, sel.sum(), round(a["w1"][0], 2), a["w1"][3], larger, S3_MAIN[k], round(a["w1 x w2"][0], 2),
                 a["w1 x w2"][3], S3_INT[k]))
s3 = pd.DataFrame(rows, columns=["cluster", "n", "regressor F", "p", "TP larger", "paper sig", "reg x bin F", "p int", "paper int sig"])
s3["main agrees"] = ((s3.p < 0.05) & s3["TP larger"]) == s3["paper sig"]
s3["int agrees"] = (s3["p int"] < 0.05) == s3["paper int sig"]
print(s3.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("clusters agreeing on both terms:", int((s3["main agrees"] & s3["int agrees"]).sum()), "of 7")

# The paper's ANOVA has 355 error df (362 units), two fewer than the 364 units here. For each
# cluster, the single unit whose removal brings the regressor F closest to Table S3's value:
S3_F = {"k1": 0.44, "k3": 9.35, "k4": 6.72, "k5": 2.02, "k6": 6.28, "k7": 16.89, "k8": 4.29}
drop_rows = []
names7 = np.concatenate([names[R][cl == k] for k in SEVEN])     # unit names in the order of TP, FO
for k in SEVEN:
    idx_k = np.flatnonzero(grp == k)
    full = rrf.mixed_anova(np.stack([TP[idx_k], FO[idx_k]], 1))["w1"][0]
    best = None
    for j in idx_k:
        keep = np.setdiff1d(idx_k, [j])
        a = rrf.mixed_anova(np.stack([TP[keep], FO[keep]], 1))
        cand = (abs(a["w1"][0] - S3_F[k]), j, a["w1"][0], a["w1 x w2"][3])
        best = cand if best is None or cand[0] < best[0] else best
    drop_rows.append((k, round(full, 2), S3_F[k], names7[best[1]],
                      round(best[2], 2), round(best[3], 3)))
print(pd.DataFrame(drop_rows, columns=["cluster", "F all units", "Table S3 F", "best unit to drop",
                                       "F without it", "interaction p without it"]).to_string(index=False))

**Result: partially reproduced.** Six of the seven clusters agree with Table S3 on both terms.
The regressor main effect is significant, with peak probability larger, in exactly k3, k4, k6,
k7 and k8; but k3 also has a regressor x bin interaction (p = 0.047, paper 0.09), so the
interaction is not in k7 only. The paper's df imply two units fewer than here: removing one k3
unit and one k5 unit gives Table S3's F for those clusters (9.35 and 2.02) and its interaction
p values (0.09 and 0.79), and with that k3 unit removed the k3 interaction is not significant
(p = 0.092). Which units the paper left out, and why, is not stated.

## fig5f-fear-output-vs-peak-probability

> Across units there is a continuous negative relationship between fear output and peak threat
> probability betas (R2 = 0.43), with signaling biased toward peak threat probability, so the two
> are not signaled by distinct populations.

Per unit, the mean of each beta over the code's columns 3-14 (cue and delay, 12 bins; the
code's Figure 5F scatter); variants (f-a) the 10 cue bins and (f-b) all 14 bins.

In [ ]:
S6_R2 = {"k1": 0.64, "k3": 0.46, "k4": 0.55, "k5": 0.65, "k6": 0.55, "k7": 0.58, "k8": 0.64}
for label, sl in (("code: cue and delay (12 bins)", slice(2, 14)), ("f-a: 10 cue bins", slice(2, 12)),
                  ("f-b: all 14 bins", slice(0, 14))):
    a_, b_ = TP[:, sl].mean(1), FO[:, sl].mean(1)
    r, p = stats.pearsonr(a_, b_)
    per = {k: round(stats.pearsonr(a_[grp == k], b_[grp == k])[0] ** 2, 2) for k in SEVEN}
    print(f"{label}: r = {r:.3f}, R2 = {r * r:.3f}, p = {p:.3g}; mean TP {a_.mean():.3f}, FO {b_.mean():.3f}; "
          f"units with TP > FO {np.mean(a_ > b_):.2f}")
    print("   per-cluster R2", per, "Figure S6", S6_R2)
a_, b_ = TP[:, 2:14].mean(1), FO[:, 2:14].mean(1)
fig, ax = plt.subplots(figsize=(3.5, 3.5))
ax.scatter(a_, b_, s=6)
ax.set_xlabel("peak threat probability beta"); ax.set_ylabel("fear output beta")
plt.show()
print("plotted ranges: TP", np.round([a_.min(), a_.max()], 2), "FO", np.round([b_.min(), b_.max()], 2))

**Result: reproduced.** r = -0.656, R2 = 0.430, p = 4.2e-46 over 364 units, the paper's R2, and
the per-cluster R2 equal those of Figure S6. The mean peak probability beta (0.483) is above the
mean fear output beta (0.147), with 63% of units above the diagonal. The f-a and f-b readings
give R2 0.419 and 0.430.

## table1-outcome-anova

> Every cluster showed differential firing after shock delivery or omission, a main effect of
> trial type and/or a trial type x interval interaction.

Per cluster, the 70 250-ms bins of the 17.5 s after shock offset (the code's bins 131-200), for
danger, uncertainty shock, uncertainty omission and safety; repeated measures with trial type
and interval within units.

In [ ]:
T1_OUT = {"k1": (True, True), "k2": (True, True), "k3": (True, False), "k4": (True, True),
          "k5": (True, True), "k6": (False, True), "k7": (True, True), "k8": (True, True)}   # (interaction, main)

def outcome_anova(mm, label):
    rows = []
    for k in rrf.CLUSTERS:
        sel = cl == k
        y = np.stack([mm[t][rrf.OUTCOME][:, sel] for t in ("D", "US", "UO", "S")], axis=0).transpose(2, 0, 1)
        a = rrf.mixed_anova(y)
        rows.append((k, sel.sum(), round(a["w1 x w2"][0], 2), a["w1 x w2"][3], a["w1 x w2"][6],
                     round(a["w1"][0], 2), a["w1"][3], a["w1"][6]))
    tab = pd.DataFrame(rows, columns=["cluster", "n", "interaction F", "p int", "GG p int", "type F", "p type", "GG p type"])
    tab["matches"] = [((r["p int"] < 0.05) == T1_OUT[r["cluster"]][0]) + ((r["p type"] < 0.05) == T1_OUT[r["cluster"]][1])
                      for _, r in tab.iterrows()]
    tab["any effect"] = (tab["p int"] < 0.05) | (tab["p type"] < 0.05)
    print(label); print(tab.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
    print("  clusters with an effect:", int(tab["any effect"].sum()), "of 8; tests matching Table 1:",
          int(tab["matches"].sum()), "of 16")
    return tab

t1out = outcome_anova(m250, "code normalization")
for nm in ("z-a", "z-b"):
    outcome_anova(NORMS[nm], nm)

**Result: reproduced.** Under the code's normalization every cluster has at least one effect and
the pattern of significance matches Table 1 in all 16 tests (k3 interaction only, k6 main effect
only, both in the others). With the Greenhouse-Geisser correction the interaction of k1 and k3
is no longer significant (p = 0.23 and 0.28); the Methods' normalizations leave k3 without an
effect (z-a 15 of 16, z-b 14 of 16 tests match).

## Outcome firing (Figure 6, Table S4)

The shared method of the fig6 claims: per unit, mean normalized firing per trial type in the
cluster's gray-box interval of Figure 6 (claims file: 0 to 8 s after shock offset for k1, k3,
k5 and k6, 250-ms bins 131-162; the whole 17.5 s for k2, k4, k7 and k8, bins 131-200);
studentized bootstrap CIs per cluster of each trial type and of the differences uncertainty
shock - danger, uncertainty omission - safety and danger - safety. The code holds no Table S4
computation; its outcome windows are the first 2 s (bins 131-138) and the last 10 s (bins
161-200), shown as variants. The next cell scans every window that starts at shock offset
for the one whose four trial-type means come closest to Table S4: for k1, k3, k5 and k6 it is
0 to 4 s, not the 0 to 8 s of the gray boxes, so 0 to 4 s is shown as a variant too.

In [ ]:
TYPES = ("D", "US", "UO", "S")
DIFFS = (("US-D", "US", "D"), ("UO-S", "UO", "S"), ("D-S", "D", "S"))
TABLE_S4 = {  # mean, tolerance per trial type (claims file)
    "k1": [(0.19, 0.18), (0.20, 0.17), (-0.02, 0.12), (0.15, 0.10)],
    "k2": [(0.33, 0.05), (-0.18, 0.07), (-0.05, 0.053), (0.06, 0.05)],
    "k3": [(0.07, 0.27), (-0.03, 0.21), (0.20, 0.23), (-0.18, 0.11)],
    "k4": [(0.19, 0.06), (0.25, 0.06), (-0.16, 0.04), (0.20, 0.05)],
    "k5": [(0.04, 0.11), (-0.05, 0.11), (0.25, 0.08), (-0.07, 0.067)],
    "k6": [(-0.17, 0.09), (-0.32, 0.10), (0.02, 0.07), (0.26, 0.07)],
    "k7": [(0.29, 0.06), (0.42, 0.08), (0.08, 0.05), (-0.28, 0.06)],
    "k8": [(-0.11, 0.06), (0.32, 0.07), (0.18, 0.04), (-0.02, 0.04)],
}
PAPER_PATTERN = {  # CI sign of D, US, UO, S, then US-D, UO-S, D-S (claims file criteria)
    "k1": (1, 1, 0, 1, 0, -1, 0), "k2": (1, -1, 0, 1, -1, -1, 1), "k3": (0, 0, 0, -1, 0, 1, 0),
    "k4": (1, 1, -1, 1, 0, -1, 0), "k5": (0, 0, 1, -1, 0, 1, 0), "k6": (-1, -1, 0, 1, -1, -1, -1),
    "k7": (1, 1, 1, -1, 1, 1, 1), "k8": (-1, 1, 1, 0, 1, 1, -1),
}

def outcome_cis(k, mm, window, gen):
    sel = cl == k
    per = {t: np.nanmean(mm[t][window][:, sel], 0) for t in TYPES}
    cis = {t: rrf.boot_ci(per[t], gen) for t in TYPES}
    for nm, a, b in DIFFS:
        cis[nm] = rrf.boot_ci(per[a] - per[b], gen)
    return cis

def fig6_table(k, gen, mm=None, window=None):
    mm = m250 if mm is None else mm
    window = rrf.FIG6_WINDOW[k] if window is None else window
    cis = outcome_cis(k, mm, window, gen)
    rows = []
    for j, key in enumerate(TYPES + tuple(d[0] for d in DIFFS)):
        ci = cis[key]
        ref = TABLE_S4[k][j] if j < 4 else None
        rows.append((key, round(ci[0], 3), round(ci[1], 3), round(ci[2], 3), rrf.ci_sign(ci), PAPER_PATTERN[k][j],
                     rrf.ci_sign(ci) == PAPER_PATTERN[k][j], ref[0] if ref else np.nan,
                     bool(abs(ci[0] - ref[0]) <= ref[1]) if ref else np.nan))
    tab = pd.DataFrame(rows, columns=["measure", "mean", "low", "high", "CI sign", "paper sign", "agrees",
                                      "Table S4 mean", "within tol"])
    return tab

def fig6_report(k, claim):
    tab = fig6_table(k, seeded(claim))
    print(f"{k}, window {rrf.FIG6_WINDOW[k]}, n = {(cl == k).sum()}")
    print(tab.to_string(index=False))
    print(f"  CIs agreeing: {int(tab.agrees.sum())} of 7; means within tolerance: {int(tab['within tol'].iloc[:4].sum())} of 4")
    for label, win in (("other window: first 2 s", rrf.b250(131, 138)), ("other window: last 10 s", rrf.b250(161, 200)),
                       ("other window: 0-8 s" if rrf.FIG6_WINDOW[k] == rrf.OUTCOME else "other window: 17.5 s",
                        rrf.OUTCOME_0_8 if rrf.FIG6_WINDOW[k] == rrf.OUTCOME else rrf.OUTCOME)):
        tv = fig6_table(k, seeded(claim), window=win)
        print(f"  {label}: means {dict(zip(tv.measure, tv['mean']))}; CIs agreeing {int(tv.agrees.sum())} of 7")
    if rrf.FIG6_WINDOW[k] == rrf.OUTCOME_0_8:
        tv = fig6_table(k, seeded(claim), window=rrf.b250(131, 146))
        print(f"  variant window 0-4 s after shock offset: means {dict(zip(tv.measure, tv['mean']))}; CIs agreeing "
              f"{int(tv.agrees.sum())} of 7; means within tolerance {int(tv['within tol'].iloc[:4].sum())} of 4")
    for nm in ("z-a", "z-b"):
        tv = fig6_table(k, seeded(claim), mm=NORMS[nm])
        print(f"  {nm}: CIs agreeing {int(tv.agrees.sum())} of 7, means within {int(tv['within tol'].iloc[:4].sum())} of 4")
    return tab

In [ ]:
# Which window starting at shock offset reproduces the Table S4 means? (largest absolute difference
# over the four trial types)
scan = []
for k in rrf.CLUSTERS:
    ref = np.array([m for m, _ in TABLE_S4[k]])
    best = min(((np.abs(np.array([np.nanmean(m250[t][130:b][:, cl == k]) for t in TYPES]) - ref).max(), (b - 130) / 4)
                for b in range(134, 201)), key=lambda x: x[0])
    v8 = np.abs(np.array([np.nanmean(m250[t][rrf.OUTCOME_0_8][:, cl == k]) for t in TYPES]) - ref).max()
    v17 = np.abs(np.array([np.nanmean(m250[t][rrf.OUTCOME][:, cl == k]) for t in TYPES]) - ref).max()
    scan.append((k, best[1], round(best[0], 4), round(v8, 3), round(v17, 3)))
print(pd.DataFrame(scan, columns=["cluster", "best window end (s)", "max |diff| there", "max |diff| 0-8 s",
                                  "max |diff| 0-17.5 s"]).to_string(index=False))

## fig6a-k1-shock

> k1 units are generally shock responsive, with equivalent, transient firing after danger and
> uncertainty shock.

In [ ]:
f6 = {}
f6["k1"] = fig6_report("k1", "fig6a-k1-shock")

**Result: not reproduced.** In the claims file's window, 0 to 8 s after shock offset, the danger
and uncertainty shock CIs include zero (means 0.085 and 0.053), where the paper's are above zero;
five of seven CIs agree and the four means are within the (wide) tolerances. The criterion's
fallback needs both shock CIs above zero, so the claim is not reproduced in this window. In the
window that reproduces Table S4's means, 0 to 4 s after shock offset, all seven CIs agree and
the four means are within tolerance (0.191, 0.208, -0.012, 0.147): the result depends on which
window the paper analysed (see the window scan above).

## fig6b-k2-anti-ppe

> k2 units fire most to the predicted shock on danger trials and are inhibited by the surprising
> shock on uncertainty trials, an anti-positive prediction error signal.

In [ ]:
f6["k2"] = fig6_report("k2", "fig6b-k2-anti-ppe")

**Result: partially reproduced.** Danger is above zero, uncertainty shock below zero, and shock -
danger below zero, with all four means within tolerance; six of seven CIs agree. The omission CI
ends just below zero (-0.093 to -0.005), where the paper's just includes it (-0.09 to 0.003).

## fig6c-k3-continued-threat

> k3 units sharply reduce firing after shock but sustain excitation after omission on
> uncertainty trials, a continued threat signal.

In [ ]:
f6["k3"] = fig6_report("k3", "fig6c-k3-continued-threat")

**Result: reproduced.** All seven CIs agree with the paper's pattern and the four means are within
tolerance in the 0 to 8 s window; in the 0 to 4 s window six of seven agree and the four means
are within 0.01 of Table S4's.

## fig6de-k4-k5-persistent-threat

> k4 units sustain inhibition and k5 units sustain excitation on uncertainty omission trials,
> persistent threat signaling through opposite firing directions.

In [ ]:
f6["k4"] = fig6_report("k4", "fig6de-k4-k5-persistent-threat")
f6["k5"] = fig6_report("k5", "fig6de-k4-k5-persistent-threat")

**Result: partially reproduced.** All CIs of both clusters agree with the paper's pattern, and
the four k4 means are within tolerance; the k5 omission mean in the 0 to 8 s window is 0.156
against 0.25 +/- 0.08, so not every mean is within tolerance. In the 0 to 4 s window the k5
means are within tolerance (omission 0.248) and six of seven k5 CIs agree.

## fig6f-k6-ppe-then-valence

> k6 units show early positive prediction error through firing decreases (more inhibition after
> uncertainty shock than danger shock), giving way to a sustained valence signal with most firing
> to safety, little after omission and sustained inhibition after danger and uncertainty shock.

In [ ]:
f6["k6"] = fig6_report("k6", "fig6f-k6-ppe-then-valence")
print("k6 means over the whole 17.5 s:", dict(zip(TYPES, [round(float(np.nanmean(m250[t][rrf.OUTCOME][:, cl == "k6"])), 3) for t in TYPES])))

**Result: reproduced.** All seven CIs agree (danger and shock below zero, omission including zero,
safety above, all three differences below zero) and the four means are within tolerance; over
the whole 17.5 s safety still has the largest mean (0.247) and danger and shock stay below zero.

## fig6g-k7-valence-then-ppe

> k7 units show initial valence signaling (danger shock = uncertainty shock > omission > safety)
> giving way to positive prediction error (uncertainty shock > danger shock), and sustain
> inhibition to safety through the outcome period.

In [ ]:
f6["k7"] = fig6_report("k7", "fig6g-k7-valence-then-ppe")

**Result: reproduced.** All seven CIs agree and the four means equal Table S4's to two decimals
(0.290, 0.423, 0.079, -0.278).

## fig6h-k8-ppe-then-unsigned

> k8 units show early positive prediction error (uncertainty shock > danger shock) giving way to
> an unsigned prediction error, with equal firing increases on uncertainty shock and omission
> trials exceeding those for danger and safety.

In [ ]:
f6["k8"] = fig6_report("k8", "fig6h-k8-ppe-then-unsigned")

**Result: reproduced.** All seven CIs agree and the four means equal Table S4's to two decimals
(-0.107, 0.326, 0.178, -0.020).

## Summary

| Claim | Outcome |
|---|---|
| units-and-sessions | reproduced |
| cue-responsive-units | reproduced |
| behavior-discrimination | reproduced |
| kmeans-elbow | reproduced (resting on an assumption) |
| kmeans-clusters | reproduced (resting on an assumption) |
| cluster-firing-characteristics | reproduced (resting on an assumption) |
| k1-k3-low-baseline | reproduced (resting on an assumption) |
| table1-cue-interaction | reproduced (resting on an assumption) |
| fig3b-excitatory-clusters | reproduced (resting on an assumption) |
| fig3d-inhibitory-clusters | reproduced (resting on an assumption) |
| figs4-onset-and-ramp | reproduced (resting on an assumption) |
| fig4abc-threat-bias | partially reproduced (resting on an assumption) |
| fig4def-danger-uncertainty-correlation | reproduced (resting on an assumption) |
| fig4ghi-danger-safety-correlation | reproduced (resting on an assumption) |
| fig4-correlations-differ | reproduced (resting on an assumption) |
| fig5a-tuning-peaks-exaggerated | reproduced (resting on an assumption) |
| fig5a-tuning-peaks-inhibited | reproduced (resting on an assumption) |
| fig5a-tuning-anova | reproduced (resting on an assumption) |
| fig5e-peak-probability-over-fear-output | reproduced (resting on an assumption) |
| fig5-regressor-cluster | reproduced (resting on an assumption) |
| table-s3-cluster-regression | partially reproduced (resting on an assumption) |
| fig5f-fear-output-vs-peak-probability | reproduced (resting on an assumption) |
| table1-outcome-anova | reproduced (resting on an assumption) |
| fig6a-k1-shock | not reproduced (resting on an assumption) |
| fig6b-k2-anti-ppe | partially reproduced (resting on an assumption) |
| fig6c-k3-continued-threat | reproduced (resting on an assumption) |
| fig6de-k4-k5-persistent-threat | partially reproduced (resting on an assumption) |
| fig6f-k6-ppe-then-valence | reproduced (resting on an assumption) |
| fig6g-k7-valence-then-ppe | reproduced (resting on an assumption) |
| fig6h-k8-ppe-then-unsigned | reproduced (resting on an assumption) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The data set includes the paper's MATLAB code, and the
analysis follows it wherever the claims file left a reading open: the trial-level z-scores the
code computes (the Methods describe z-scoring trial-type means; the claims file's readings z-a
and z-b of the Methods are shown as variants, under which the code's k-means run no longer gives
the paper's clusters; every claim that uses normalized firing or the clusters is therefore
marked as resting on the assumption that the code is what the paper did), the code's k-means matrix (outcome epochs
from 13 s after cue onset) and its single k-means run, reproduced outside MATLAB so that the
clusters are the paper's own, the cluster-specific tuning periods, and the sign inversion of
k4, k6 and k8. Statistics are recomputed in Python: repeated-measures ANOVAs from sums of
squares (type III for the within effect of the mixed designs, as SPSS), studentized bootstrap
CIs with 1,000 resamples (their bounds move slightly with the random draw), Meng's Z. The
outcome windows of Figure 6 are those the claims file reads from the gray boxes; for k1, k3, k5
and k6 Table S4's values come from 0 to 4 s, not 0 to 8 s.

What was found about the data set. It holds every unit, session and rat the paper reports, and
the authors' derived file (cube.mat) agrees exactly with the spike and event times of the unit
files. Every count, F value and correlation of the paper that depends only on the data and the
code was recovered, most to the printed digits. Not stated in the data set's description: the
recordings cover only 20 s before to about 33.5 s after each cue (recording paused between
trials); the event channel names (only in code comments); the cluster labels (fixed only by
the code's k-means run) and the electrode locations (a table in the code). The description
calls `wave` a mean waveform per trial type (it is one per unit) and `poke.base` a 2-s baseline
(it is 20 s). The paper's Figure 5E and Table S3 ANOVAs leave out two units that the paper does
not name.